<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_5_RepresentationGeometry/student/Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_5_RepresentationGeometry/student/Lab.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Lab: Representation Geometry in Deep Learning

**Session 5: Representation Geometry**

*Static geometry, abstraction, and dynamics*

This guided lab separates three inferential targets. RSA compares pairwise geometry, CCGP tests whether a selected variable transfers across contexts, and DSA compares temporal evolution.

## Audience, prerequisites, and goals

**Audience.** Graduate students who know basic linear algebra, neural networks, and Python.

**Prerequisites.** NumPy arrays, PyTorch modules, linear classification, and matrix multiplication. No prior RSA, CCGP, or DSA experience is assumed.

By the end of the lab, you will be able to:

- construct and compare RDMs with RSA;
- distinguish within-context decodability from cross-context generalization;
- compute CCGP for a crossed-variable design;
- fit a linear evolution operator and compute a two-dimensional DSA distance;
- match RSA, CCGP, and DSA results to the claims they support.

## 1. Setup and data

The scikit-learn digits dataset ships with the library, so this notebook requires no download.

In [ ]:
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'figure.figsize': (7.2, 4.2),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})
COLORS = {'blue': '#277DA1', 'green': '#3A7D44', 'coral': '#D1495B', 'gold': '#E09F3E'}

print(f'NumPy {np.__version__} | PyTorch {torch.__version__}')

In [ ]:
X, digit = load_digits(return_X_y=True)
X = X.astype(np.float32) / 16.0
parity = digit % 2
magnitude = (digit >= 5).astype(int)
all_idx = np.arange(len(digit))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.30, stratify=digit, random_state=SEED
)

print(f'Samples: {len(X)} | train: {len(train_idx)} | test: {len(test_idx)}')
print('Factor definitions: parity = even/odd; magnitude = 0-4/5-9')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8.2, 3.6))
for d, ax in enumerate(axes.flat):
    sample = np.flatnonzero(digit == d)[0]
    ax.imshow(X[sample].reshape(8, 8), cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f'{d}: {"even" if d % 2 == 0 else "odd"}, {"small" if d < 5 else "large"}')
    ax.axis('off')
plt.suptitle('One example per digit condition', y=1.02)
plt.tight_layout()
plt.show()

### Three analysis designs

- **RSA:** ten digit conditions define a static representational geometry.
- **CCGP:** four cells formed by parity crossed with magnitude define a transfer test.
- **DSA:** a separate toy system supplies ordered trajectories and known transition rules.

Parity and magnitude are pedagogical labels for this exercise. They are not claims about the natural semantics of the dataset.

## 2. Representational similarity analysis

We first need one response vector for each condition.

### Exercise: Compute condition means

Complete a function that returns one mean response vector for each requested condition.

Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# def condition_means(representations, labels, conditions):
#     means = []
#     for condition in conditions:
#         ...
#     return np.stack(means)

In [ ]:
def condition_means(representations, labels, conditions):
    """Return one mean representation per condition, in the given order."""
    means = []
    for condition in conditions:
        mask = labels == condition
        if not np.any(mask):
            raise ValueError(f'No samples for condition {condition}')
        means.append(representations[mask].mean(axis=0))
    return np.stack(means)

### Representation 1: pixels

Pixels are already a 64-dimensional representation. We estimate each digit mean using only the held-out test split.

In [ ]:
digit_conditions = np.arange(10)
pixel_digit_means = condition_means(X[test_idx], digit[test_idx], digit_conditions)
print('Condition-by-feature shape:', pixel_digit_means.shape)

### Exercise: Build an RDM

Implement Euclidean distances between every pair of condition means. The result must be a symmetric square matrix with a zero diagonal.

Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# Hint: scipy.spatial.distance.pdist returns the unique pairwise distances.
# def euclidean_rdm(means):
#     ...

In [ ]:
def euclidean_rdm(means):
    """Return the Euclidean representational dissimilarity matrix."""
    means = np.asarray(means)
    if means.ndim != 2:
        raise ValueError('means must have shape (conditions, features)')
    return squareform(pdist(means, metric='euclidean'))

In [ ]:
pixel_rdm = euclidean_rdm(pixel_digit_means)
assert pixel_rdm.shape == (10, 10)
assert np.allclose(pixel_rdm, pixel_rdm.T)
assert np.allclose(np.diag(pixel_rdm), 0)

fig, ax = plt.subplots(figsize=(4.6, 4.0))
image = ax.imshow(pixel_rdm, cmap='magma')
ax.set(title='Pixel-space digit RDM', xlabel='digit', ylabel='digit', xticks=range(10), yticks=range(10))
fig.colorbar(image, ax=ax, label='Euclidean distance', fraction=0.046)
plt.tight_layout()
plt.show()

### Hypothesis RDMs

A binary model RDM assigns 0 to condition pairs that share a label and 1 to pairs that differ. We will compare the observed digit RDM with parity and magnitude hypotheses.

In [ ]:
def binary_model_rdm(condition_labels):
    labels = np.asarray(condition_labels)
    return (labels[:, None] != labels[None, :]).astype(float)

parity_model_rdm = binary_model_rdm(digit_conditions % 2)
magnitude_model_rdm = binary_model_rdm(digit_conditions >= 5)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2))
for ax, matrix, title in zip(
    axes, [parity_model_rdm, magnitude_model_rdm], ['Parity model RDM', 'Magnitude model RDM']
):
    ax.imshow(matrix, cmap='Greys', vmin=0, vmax=1)
    ax.set(title=title, xlabel='digit', ylabel='digit', xticks=range(10), yticks=range(10))
plt.tight_layout()
plt.show()

### Exercise: Compare two RDMs

Extract the same upper-triangle entries from both RDMs and return their Spearman correlation.

Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# def rsa_score(observed_rdm, model_rdm):
#     ...

In [ ]:
def rsa_score(observed_rdm, model_rdm):
    """Return Spearman correlation between matching RDM upper triangles."""
    observed_rdm = np.asarray(observed_rdm)
    model_rdm = np.asarray(model_rdm)
    if observed_rdm.shape != model_rdm.shape:
        raise ValueError('RDMs must have matching shapes')
    upper = np.triu_indices_from(observed_rdm, k=1)
    return float(spearmanr(observed_rdm[upper], model_rdm[upper]).statistic)

In [ ]:
print(f'Pixels vs parity model:    {rsa_score(pixel_rdm, parity_model_rdm): .3f}')
print(f'Pixels vs magnitude model: {rsa_score(pixel_rdm, magnitude_model_rdm): .3f}')

**Interpretation.** The score measures agreement in the ordering of pairwise distances. It does not establish corresponding units, identical computations, or causal equivalence.

## 3. Train matched neural networks

Both models use the same 64-64-16 encoder. Their output objectives differ.

In [ ]:
class DigitMLP(torch.nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(64, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 16),
            torch.nn.ReLU(),
        )
        self.head = torch.nn.Linear(16, output_dim)

    def forward(self, inputs):
        representation = self.encoder(inputs)
        return self.head(representation), representation

def fit_model(task, epochs=150):
    torch.manual_seed(SEED)
    output_dim = 10 if task == 'digit' else 2
    model = DigitMLP(output_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    inputs = torch.tensor(X[train_idx])
    if task == 'digit':
        targets = torch.tensor(digit[train_idx], dtype=torch.long)
        loss_fn = torch.nn.CrossEntropyLoss()
    else:
        targets = torch.tensor(
            np.column_stack([parity[train_idx], magnitude[train_idx]]), dtype=torch.float32
        )
        loss_fn = torch.nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()
    return model.eval()

def get_representations(model, inputs):
    with torch.no_grad():
        _, representation = model(torch.tensor(inputs, dtype=torch.float32))
    return representation.numpy()

In [ ]:
torch.manual_seed(SEED)
untrained_model = DigitMLP(output_dim=10).eval()
digit_model = fit_model('digit')
factor_model = fit_model('factors')

representations = {
    'pixels': X,
    'untrained': get_representations(untrained_model, X),
    'digit-trained': get_representations(digit_model, X),
    'factor-trained': get_representations(factor_model, X),
}

with torch.no_grad():
    digit_logits, _ = digit_model(torch.tensor(X[test_idx]))
    factor_logits, _ = factor_model(torch.tensor(X[test_idx]))
digit_test_accuracy = accuracy_score(digit[test_idx], digit_logits.argmax(1).numpy())
factor_predictions = (factor_logits.numpy() > 0).astype(int)
parity_test_accuracy = accuracy_score(parity[test_idx], factor_predictions[:, 0])
magnitude_test_accuracy = accuracy_score(magnitude[test_idx], factor_predictions[:, 1])

print(f'Digit objective, digit test accuracy:       {digit_test_accuracy:.3f}')
print(f'Factor objective, parity test accuracy:      {parity_test_accuracy:.3f}')
print(f'Factor objective, magnitude test accuracy:   {magnitude_test_accuracy:.3f}')

The task accuracies provide evidence that each network learned its assigned outputs. They do not yet describe the organization of the hidden representation.

In [ ]:
digit_rdms = {}
for name, values in representations.items():
    means = condition_means(values[test_idx], digit[test_idx], digit_conditions)
    digit_rdms[name] = euclidean_rdm(means)

fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.0))
for ax, (name, matrix) in zip(axes, digit_rdms.items()):
    normalized = matrix / matrix.max()
    ax.imshow(normalized, cmap='magma', vmin=0, vmax=1)
    ax.set(title=name, xlabel='digit', ylabel='digit', xticks=range(10), yticks=range(10))
plt.suptitle('Digit RDMs, each normalized for display', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
rsa_results = {}
for name, matrix in digit_rdms.items():
    rsa_results[name] = {
        'parity': rsa_score(matrix, parity_model_rdm),
        'magnitude': rsa_score(matrix, magnitude_model_rdm),
    }

print(f'{"representation":<17} {"parity RSA":>11} {"magnitude RSA":>14}')
for name, scores in rsa_results.items():
    print(f'{name:<17} {scores["parity"]:>11.3f} {scores["magnitude"]:>14.3f}')

In [ ]:
names = list(rsa_results)
x = np.arange(len(names))
width = 0.36
fig, ax = plt.subplots(figsize=(8.0, 3.8))
ax.bar(x - width / 2, [rsa_results[n]['parity'] for n in names], width, label='parity', color=COLORS['blue'])
ax.bar(x + width / 2, [rsa_results[n]['magnitude'] for n in names], width, label='magnitude', color=COLORS['gold'])
ax.axhline(0, color='#66717E', linewidth=0.8)
ax.set(ylabel='Spearman RSA score', xticks=x, xticklabels=names, title='Agreement with factor model RDMs')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### View the four crossed condition means

PCA is used only for visualization. RSA and CCGP are computed in the full representation space.

In [ ]:
factor_cell = 2 * magnitude + parity
cell_names = ['small even', 'small odd', 'large even', 'large odd']
fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.1))
for ax, (name, values) in zip(axes, representations.items()):
    held_out = values[test_idx]
    projected = PCA(n_components=2).fit_transform(held_out)
    means_2d = condition_means(projected, factor_cell[test_idx], np.arange(4))
    for cell, point in enumerate(means_2d):
        is_even = cell % 2 == 0
        is_small = cell < 2
        ax.scatter(
            point[0], point[1], s=80,
            color=COLORS['blue'] if is_even else COLORS['coral'],
            marker='o' if is_small else '^',
        )
        ax.annotate(cell_names[cell], point, xytext=(4, 4), textcoords='offset points', fontsize=8)
    ax.set(title=name, xlabel='PC 1', ylabel='PC 2')
plt.suptitle('PCA view of held-out parity-by-magnitude means', y=1.04)
plt.tight_layout()
plt.show()

## 4. Cross-condition generalization

A within-context decoder asks whether a target is locally separable. CCGP asks whether a decoder trained in one context transfers to the other context.

In [ ]:
def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight='balanced', solver='liblinear', max_iter=1000, random_state=SEED
        ),
    )

def within_context_score(values, target, context):
    scores = []
    for context_value in (0, 1):
        train_mask = train_idx[context[train_idx] == context_value]
        test_mask = test_idx[context[test_idx] == context_value]
        probe = make_probe().fit(values[train_mask], target[train_mask])
        prediction = probe.predict(values[test_mask])
        scores.append(balanced_accuracy_score(target[test_mask], prediction))
    return float(np.mean(scores)), scores

### Exercise: Compute bidirectional CCGP

For each training context, fit a probe on the training split and evaluate it on the opposite context in the test split. Return the mean and the two directional scores.

Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# def ccgp(values, target, context):
#     scores = []
#     for train_context in (0, 1):
#         ...
#     return float(np.mean(scores)), scores

In [ ]:
def ccgp(values, target, context):
    """Balanced-accuracy transfer between two held-out contexts."""
    scores = []
    for train_context in (0, 1):
        train_mask = train_idx[context[train_idx] == train_context]
        test_mask = test_idx[context[test_idx] != train_context]
        probe = make_probe().fit(values[train_mask], target[train_mask])
        prediction = probe.predict(values[test_mask])
        scores.append(balanced_accuracy_score(target[test_mask], prediction))
    return float(np.mean(scores)), scores

In [ ]:
ccgp_results = {}
within_results = {}
for name, values in representations.items():
    within_results[name] = {
        'parity': within_context_score(values, parity, magnitude)[0],
        'magnitude': within_context_score(values, magnitude, parity)[0],
    }
    ccgp_results[name] = {
        'parity': ccgp(values, parity, magnitude)[0],
        'magnitude': ccgp(values, magnitude, parity)[0],
    }

print(f'{"representation":<17} {"parity within":>13} {"parity CCGP":>12} {"magnitude within":>16} {"magnitude CCGP":>15}')
for name in representations:
    print(
        f'{name:<17} {within_results[name]["parity"]:>13.3f} '
        f'{ccgp_results[name]["parity"]:>12.3f} '
        f'{within_results[name]["magnitude"]:>16.3f} '
        f'{ccgp_results[name]["magnitude"]:>15.3f}'
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7), sharey=True)
for ax, target_name in zip(axes, ['parity', 'magnitude']):
    x = np.arange(len(names))
    ax.bar(x - width / 2, [within_results[n][target_name] for n in names], width, label='within context', color=COLORS['gold'])
    ax.bar(x + width / 2, [ccgp_results[n][target_name] for n in names], width, label='CCGP', color=COLORS['green'])
    ax.axhline(0.5, color='#66717E', linestyle=':', label='chance' if target_name == 'parity' else None)
    ax.set(title=target_name, xticks=x, xticklabels=names, ylim=(0, 1.05), ylabel='balanced accuracy')
    ax.tick_params(axis='x', rotation=25)
axes[0].legend(frameon=False, loc='lower left')
plt.suptitle('Local decodability and cross-context transfer', y=1.04)
plt.tight_layout()
plt.show()

**Interpretation prompt.** Find a representation with high within-context accuracy but lower CCGP. What information would be lost if only within-context accuracy were reported?

## 5. Dynamical similarity analysis

RSA and CCGP above use static examples. DSA requires trajectories because its target is the rule that maps one state to the next. We use a fully observed linear system, $\mathbf{z}_{t+1}=A\mathbf{z}_t$, as a transparent special case of DSA.

In [ ]:
assert ccgp_results['factor-trained']['parity'] > ccgp_results['digit-trained']['parity'] + 0.20
assert ccgp_results['factor-trained']['magnitude'] > ccgp_results['digit-trained']['magnitude'] + 0.20
print('CCGP sanity checks passed for this fixed-seed run.')

### Three transition rules on one orbit

All three systems visit the same 31 points on the unit circle. The base system advances by one point, the reflected system expresses the same dynamics in a reflected coordinate system, and the fast system advances by three points. The state set alone cannot identify which transition rule generated it.

In [ ]:
def rotation(angle):
    return np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])

def simulate_linear(operator, initial_state, steps):
    states = [np.asarray(initial_state, dtype=float)]
    for _ in range(steps):
        states.append(operator @ states[-1])
    return np.stack(states)

n_states = 31
step_angle = 2 * np.pi / n_states
base_operator = rotation(step_angle)
reflection = np.diag([1.0, -1.0])
reflected_operator = reflection @ base_operator @ reflection.T
fast_operator = rotation(3 * step_angle)
initial_state = np.array([1.0, 0.0])

trajectories = {
    'base': simulate_linear(base_operator, initial_state, n_states),
    'reflected coordinates': simulate_linear(reflected_operator, reflection @ initial_state, n_states),
    'different speed': simulate_linear(fast_operator, initial_state, n_states),
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.2))
for ax, (name, states) in zip(axes, trajectories.items()):
    ax.scatter(states[:-1, 0], states[:-1, 1], s=16, color='#AAB2B9')
    starts = states[:7]
    changes = states[1:8] - starts
    ax.quiver(starts[:, 0], starts[:, 1], changes[:, 0], changes[:, 1], angles='xy', scale_units='xy', scale=1, color=COLORS['green'], width=0.012)
    ax.set(title=name, xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), aspect='equal', xlabel='$z_1$', ylabel='$z_2$')
plt.suptitle('The same state set can support different transition rules', y=1.04)
plt.tight_layout()
plt.show()

### Estimate the evolution operator

Dynamic mode decomposition estimates the matrix that best maps current states to next states. In this linear, fully observed example, ordinary least squares recovers the operator directly. Full DSA uses delay embeddings before estimating operators for nonlinear systems.

In [ ]:
def fit_linear_dynamics(states):
    """Fit A in z[t + 1] = A z[t] by least squares."""
    current = states[:-1].T
    future = states[1:].T
    return future @ np.linalg.pinv(current)

estimated_operators = {name: fit_linear_dynamics(states) for name, states in trajectories.items()}
for name, operator in estimated_operators.items():
    print(f'{name:<22} eigenvalues: {np.round(np.linalg.eigvals(operator), 3)}')

### Exercise: Compute a two-dimensional DSA distance

Search over two-dimensional orthogonal matrices $Q$ and return $\min_Q\lVert A-QBQ^\top\rVert_F$. Try the TODO cell, then reveal the reference solution.

In [ ]:
# TODO: implement this function.
# Include both rotations and reflections in O(2).
# def dsa_distance_2d(operator_a, operator_b, n_angles=720):
#     best_distance = np.inf
#     best_alignment = None
#     ...
#     return best_distance, best_alignment

In [ ]:
def dsa_distance_2d(operator_a, operator_b, n_angles=720):
    """Align two 2D linear operators over the orthogonal group O(2)."""
    best_distance = np.inf
    best_alignment = None
    for determinant in (1.0, -1.0):
        handedness = np.diag([1.0, determinant])
        for angle in np.linspace(0, 2 * np.pi, n_angles, endpoint=False):
            alignment = rotation(angle) @ handedness
            aligned_b = alignment @ operator_b @ alignment.T
            distance = np.linalg.norm(operator_a - aligned_b, ord='fro')
            if distance < best_distance:
                best_distance = distance
                best_alignment = alignment
    return float(best_distance), best_alignment

In [ ]:
dynamic_names = list(estimated_operators)
dsa_matrix = np.zeros((len(dynamic_names), len(dynamic_names)))
for i, name_i in enumerate(dynamic_names):
    for j, name_j in enumerate(dynamic_names):
        dsa_matrix[i, j] = dsa_distance_2d(estimated_operators[name_i], estimated_operators[name_j])[0]

fig, ax = plt.subplots(figsize=(5.4, 4.2))
image = ax.imshow(dsa_matrix, cmap='viridis')
ax.set(title='DSA distance between fitted operators', xticks=range(3), yticks=range(3), xticklabels=dynamic_names, yticklabels=dynamic_names)
ax.tick_params(axis='x', rotation=25)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{dsa_matrix[i, j]:.3f}', ha='center', va='center', color='white' if dsa_matrix[i, j] > 0.3 else 'black')
fig.colorbar(image, ax=ax, label='aligned Frobenius distance', fraction=0.046)
plt.tight_layout()
plt.show()

assert dsa_matrix[0, 1] < 1e-6
assert dsa_matrix[0, 2] > 0.50
print('DSA sanity checks passed: reflection aligns; the different transition speed remains distinct.')

## 6. Reconcile the measurements

The three methods now support three different statements:

- **RSA:** the factor-trained digit representation has stronger agreement with the parity and magnitude model RDMs in this run.
- **CCGP:** its linear factor readouts transfer more accurately across the held-out context in this run.
- **DSA:** the base and reflected systems have equivalent fitted dynamics after orthogonal alignment, while the different transition speed remains separated.

In [ ]:
base_orbit = trajectories['base'][:-1]
fast_orbit = trajectories['different speed'][:-1]
nearest = np.argmin(((base_orbit[:, None, :] - fast_orbit[None, :, :]) ** 2).sum(axis=2), axis=1)
fast_matched_by_state = fast_orbit[nearest]
orbit_rsa = rsa_score(euclidean_rdm(base_orbit), euclidean_rdm(fast_matched_by_state))
dynamic_distance = dsa_matrix[0, 2]
print(f'RSA after matching the same orbit states: {orbit_rsa:.3f}')
print(f'DSA distance between transition rules:   {dynamic_distance:.3f}')

**Interpretation.** The orbit contains the same states, so its matched static geometry agrees. The temporal rule differs, which DSA detects. RSA and DSA are not competing estimates of one quantity.

## Pitfalls and extensions

**RSA correspondence.** RDM entries must refer to matched conditions. A high score does not establish matched units or dynamics.

**CCGP leakage.** Fit scaling and decoding only on the training context. Otherwise information from the held-out context enters the probe.

**DSA scope.** The exercise implements a two-dimensional linear special case. Full DSA builds delay embeddings, estimates lifted operators, and optimizes their alignment.

**Optional extension.** Add observation noise, vary trajectory length, and measure how operator estimation changes the DSA distance.

## References

- Kriegeskorte, N., Mur, M., and Bandettini, P. A. (2008). Representational similarity analysis: connecting the branches of systems neuroscience. *Frontiers in Systems Neuroscience*, 2:4.
- Diedrichsen, J., and Kriegeskorte, N. (2017). Representational models: a common framework for understanding encoding, pattern-component, and representational-similarity analysis. *PLOS Computational Biology*, 13(4):e1005508.
- Bernardi, S., Benna, M. K., Rigotti, M., Munuera, J., Fusi, S., and Salzman, C. D. (2020). The geometry of abstraction in the hippocampus and prefrontal cortex. *Cell*, 183(4), 954-967.e21.
- Ostrow, M., Eisen, A., Kozachkov, L., and Fiete, I. (2023). Beyond geometry: comparing the temporal structure of computation in neural circuits with dynamical similarity analysis. *Advances in Neural Information Processing Systems*, 36.